In [ ]:
from main import sheet_processor
import logging
import pandas as pd
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import json
import os


# How to set log level to debug
logging.basicConfig(level=logging.DEBUG)

logger = logging.getLogger(__name__)

## Model Selection

In this section, we specify which models the system will use. To simplify the initial setup and testing workflow, we currently use a single selected model across all processing components: **PII Detection, PII Reflection, Non-PII Detection, and ReadMe Detection**. This approach allows us to validate the end-to-end pipeline without introducing variability from multiple model behaviors.

At the moment, only two models are deployed through the Azure service: **GPT-4.1 Mini** and **GPT-4.1 Nano**. These models are the available choices for experimentation and integration. As the project evolves, additional models can be added or different models can be assigned to specific components for more specialized behavior.


In [2]:
MODEL = 'gpt-4.1-nano'

ALLOWED_MODELS = ['gpt-4.1-mini', 'gpt-4.1-nano']

if MODEL not in ALLOWED_MODELS:
    raise ValueError(f'Invalid model: {MODEL}. Please use one of the following: {", ".join(ALLOWED_MODELS)}')

# Dataset Selection

You can select a dataset in one of two ways:

1. **Using a download URL**: Provide the URL to a CSV, XLS, or XLSX file.
2. **Using a local file**: Choose a file from the `research/data` folder in the project.

This flexibility allows you to work with both online datasets and local files for testing and analysis.


In [3]:
file_path = 'research/data/'  # This is a local test file

# DataSampler

The `DataSampler` class is used to extract a subset of rows from the original dataset.  
This sampling helps to:

- **Reduce memory usage** when working with large datasets.
- **Increase processing speed** during classification and analysis.

By working on a smaller, representative portion of the data, we can efficiently test the pipeline and classifiers without loading the entire dataset.


In [ ]:
from utils.processing import DataSampler, create_report
import os

sampler = DataSampler()

with open('/Users/liangtelkamp/Documents/GitHub/hdx-ssd-pipeline/data/isps.json', 'r') as f:
    isp = json.load(f)

isp = isp["default"]

for file in os.listdir('research/data'):
    if not 'panama' in file:
        continue
    try:
        file_path = f'research/data/{file}'

        sdd_report = create_report(file_path)
        for sheet in sdd_report:
            print(sheet.keys())
            sheet["columns"] = sheet["columns"][:2]
            sdd_report = sheet_processor(sheet, isp, MODEL)
        with open(f'research/results/test_results/groundtruth/{file}.json', 'w') as f:
            json.dump(sdd_report, f, indent=4)
        break
    except Exception as e:
        print(f"Error processing file: {file} {e}")
        import traceback

        traceback.print_exc()
        break

dict_keys(['resource_id', 'file_name', 'file_url', 'processing_timestamp', 'processing_success', 'n_records', 'n_columns', 'completion_tokens', 'prompt_tokens', 'sheet_name', 'pii_classifier_model', 'pii_reflection_model', 'non_pii_model', 'pii_sensitive', 'non_pii_sensitive', 'columns', 'non_pii', 'error_source', 'error_message'])
Processing sheet: 
dict_keys(['resource_id', 'file_name', 'file_url', 'processing_timestamp', 'processing_success', 'n_records', 'n_columns', 'completion_tokens', 'prompt_tokens', 'sheet_name', 'pii_classifier_model', 'pii_reflection_model', 'non_pii_model', 'pii_sensitive', 'non_pii_sensitive', 'columns', 'non_pii', 'error_source', 'error_message'])


Reflecting on PII sensitivity: 100%|██████████| 2/2 [00:00<00:00, 300.56it/s]


Non-PII classification: {'resource_id': None, 'file_name': 'research/data/panama.xlsx', 'file_url': None, 'processing_timestamp': '2025-12-11 15:53:18', 'processing_success': True, 'n_records': 362, 'n_columns': 128, 'completion_tokens': 244, 'prompt_tokens': 1340, 'sheet_name': 'Data', 'pii_classifier_model': 'gpt-4.1-nano', 'pii_reflection_model': 'gpt-4.1-nano', 'non_pii_model': 'gpt-4.1-nano', 'pii_sensitive': False, 'non_pii_sensitive': False, 'columns': [{'column_name': 'RspID', 'sample_values': ['UPS-4486599582818017760', 'UPS3757161354494566516', 'UPS-3513941599311189417', 'UPS-3216076094266760559', 'UPS8561606300270506481'], 'pii': {'entity_type': 'None', 'sensitive': False}}, {'column_name': 'Country', 'sample_values': ['Panama', 'Panama', 'Panama', 'Panama', 'Panama'], 'pii': {'entity_type': 'None', 'sensitive': False}}], 'non_pii': {'sensitivity': 'NON_SENSITIVE', 'sensitive_columns': [], 'cited_isp_rules': ['Humanitarian Needs Overview (HNO) and Humanitarian Response Plan 

# Processing Each Sheet Individually

For each sheet in the dataset, we perform the following processing steps:

1. **PII Detection** – Identify columns containing personally identifiable information.
2. **PII Reflection Detection** – Detect columns that might indirectly reveal PII.
3. **Non-PII Detection** – Classify remaining columns that do not contain sensitive information.

**Special Case:**  
If a sheet is named `readme`, `instructions`, or `metadata`, we skip the column-level classification and instead perform a **simple ReadMe scan** to extract relevant information from the documentation.


NameError: name 'sheets' is not defined

# Save


In [ ]:
# Save the report in the research/results/test_results folder under the corresponding file name
folder = f'research/results/test_results/{MODEL}'
# Check if folder exists, if not create it
if not os.path.exists(folder):
    os.makedirs(folder)

file_name = f'{file_path.split("/")[-1]}.json'
save_path = os.path.join(folder, file_name)
with open(save_path, 'w') as f:
    json.dump(reports, f, indent=2)

In [ ]:
print(f'Report saved to {save_path}')

# Evaluation


In [ ]:
def compare_pii_columns(gt_reports, pred_reports):
    """Compare PII sensitivity for all columns across sheets."""
    records = []
    for gt, pred in zip(gt_reports, pred_reports):
        gt_cols = {c['column_name']: c['pii']['sensitive'] for c in gt['columns']}
        pred_cols = {c['column_name']: c['pii']['sensitive'] for c in pred['columns']}
        for col_name in gt_cols:
            records.append({'column_name': col_name, 'true': gt_cols[col_name], 'pred': pred_cols.get(col_name, False)})
    return pd.DataFrame(records)


def compare_pii_table_level(gt_reports, pred_reports):
    """Compare PII sensitivity at the table level."""
    records = []
    for gt, pred in zip(gt_reports, pred_reports):
        records.append({'true': gt.get('pii_sensitive', False), 'pred': pred.get('pii_sensitive', False)})
    return pd.DataFrame(records)


def compare_non_pii_table_level(gt_reports, pred_reports):
    """Compare non-PII sensitivity at the table level."""
    records = []
    for gt, pred in zip(gt_reports, pred_reports):
        records.append({'true': gt.get('non_pii_sensitive', False), 'pred': pred.get('non_pii_sensitive', False)})
    return pd.DataFrame(records)


def calculate_metrics(df: pd.DataFrame):
    """Compute accuracy, precision, recall, and F1 score."""
    return {
        'accuracy': accuracy_score(df['true'], df['pred']),
        'precision': precision_score(df['true'], df['pred'], zero_division=0),
        'recall': recall_score(df['true'], df['pred'], zero_division=0),
        'f1': f1_score(df['true'], df['pred'], zero_division=0),
    }

In [ ]:
filename = 'data.xlsx'

# Read the groundtruth and predictions
with open(f'research/results/test_results/groundtruth/{filename}.json', 'r') as f:
    groundtruth = json.load(f)
with open(f'research/results/test_results/gpt-4.1-nano/{filename}.json', 'r') as f:
    predictions = json.load(f)

# Calculate metrics
metrics = {
    filename: {
        'pii_columns': calculate_metrics(compare_pii_columns(groundtruth, predictions)),
        'pii_table_level': calculate_metrics(compare_pii_table_level(groundtruth, predictions)),
        'non_pii_table_level': calculate_metrics(compare_non_pii_table_level(groundtruth, predictions)),
    }
}
metrics